# QA — Reranking Results Inspection

Lightweight QA tool for inspecting reranking results.
Tables with product images, side-by-side baseline vs reranker, and a single NDCG comparison table.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

from src.data.loader import load_search_data, load_product_metadata
from src.data.cleaner import clean_search_data, clean_product_metadata
from src.models.reranker import rerank
from src.models.strategies import RerankStrategy
from src.evaluation.metrics import evaluate_against_baseline

CDN_BASE = "https://cdn.aboutstatic.com/file"

## Load data

In [ ]:
search_df = clean_search_data(load_search_data())
product_df = clean_product_metadata(load_product_metadata())
prod_lookup = product_df.set_index("product_id")

print(f"{len(search_df):,} rows | {search_df['search_term'].nunique()} terms | {len(product_df):,} products")

## Baseline vs Reranker (NDCG@10)

In [ ]:
evaluate_against_baseline(search_df, k=10)

## Inspect reranking for a query

Shows top products as the reranker orders them, with thumbnails, scores, and the original baseline rank.

In [ ]:
def image_tag(image_hash, size=64):
    if pd.isna(image_hash) or not image_hash:
        return ""
    return f"<img src='{CDN_BASE}/{image_hash}?quality=75&height={size}&width={size}' width='{size}' height='{size}'>"


def inspect(term, k=10):
    """Show reranker results with product images and baseline rank delta."""
    term_df = search_df[search_df["search_term"] == term].copy()
    if term_df.empty:
        print(f"Term '{term}' not in dataset.")
        return

    # Baseline order
    term_df = term_df.sort_values("impression_pos_avg", ascending=True).reset_index(drop=True)
    term_df["baseline_rank"] = range(1, len(term_df) + 1)

    # Reranker order
    ranked = rerank(term, search_df, strategy=RerankStrategy.SMOOTHED_CTR, top_k=k)
    reranked_map = {r["product_id"]: r for r in ranked}

    rows = []
    for r in ranked:
        pid = r["product_id"]
        prod = prod_lookup.loc[pid] if pid in prod_lookup.index else pd.Series()
        baseline_row = term_df[term_df["product_id"] == pid]
        base_rank = int(baseline_row["baseline_rank"].iloc[0]) if not baseline_row.empty else "-"
        delta = base_rank - r["rank"] if isinstance(base_rank, int) else "-"
        delta_str = f"+{delta}" if isinstance(delta, int) and delta > 0 else str(delta)

        rows.append({
            "": image_tag(prod.get("image_hash")),
            "product": prod.get("product_name", ""),
            "rerank #": r["rank"],
            "baseline #": base_rank,
            "delta": delta_str,
            "score": r["score"],
            "clicks": r["clicks"],
            "impr.": r["impressions"],
        })

    header = f"<h3><code>{term}</code> — top {k} (smoothed CTR)</h3>"
    header += "<p>delta = baseline_rank - reranker_rank (positive = moved up)</p>"
    display(HTML(header + pd.DataFrame(rows).to_html(escape=False, index=False)))


# --- Try some queries ---
inspect("kleid", k=10)

In [ ]:
inspect("barrel jeans", k=10)

In [ ]:
inspect("sneaker", k=10)

## Pick your own query

In [ ]:
# Change the term below to inspect any query
inspect("hose", k=10)